# Third-edition lever sweep

The report defines its scenarios as combinations of five levers and publishes three of them
(S0, S1, S2). This notebook runs the rest of the grid: every combination of traffic growth,
aircraft technology, operations and SAF deployment.

    3 traffic x 4 technology x 3 operations x 3 SAF = 108 runs

Market-based measures are not swept. They are computed as the residual needed to reach the
target, so they follow from the other four levers rather than varying independently.

Processes are built on the fly by `sweep.py` rather than from 108 committed configuration
files, and each one is discarded as soon as it has been reduced to the series worth keeping,
so peak memory is a single process.

**A note on SAF resolution.** F2 and F3 are published as quantities *per pathway*, so they map
onto the eleven-carrier energy files of the full edition. F1 publishes only a total volume with
no pathway breakdown, so it is modelled as a single generic SAF carrier, reusing the light
edition's S0 energy file rather than inventing a pathway split. Pathway-level outputs are
therefore undefined for the F1 arm, and comparisons across the SAF axis stay on totals.

In [ ]:
%matplotlib widget
import json
import time

import pandas as pd

from aeromaps import assemble_processes

import sweep

## Benchmark before committing to 108 runs

`compute_all` is sequential, so the grid costs 108 times one run. Time one cell first.

In [ ]:
start = time.perf_counter()
reference = sweep.build_process(*sweep.PUBLISHED_CELLS["S1"])
reference.compute()
elapsed = time.perf_counter() - start

print(f"one cell: {elapsed:.1f} s  ->  108 cells ~ {elapsed * 108 / 60:.0f} min")

## The grid reproduces the published scenario

S1 is the central/T3/O3/F2 cell of this grid, so it must come back identical to the committed
scenario output. If it does not, the lever mapping is wrong and nothing below is meaningful.

In [ ]:
committed = json.loads((sweep.ATAG / "3rd_edition_full" / "data_outputs" / "s1.json").read_text())[
    "climate_outputs"
]
computed = reference.data["climate_outputs"]
years = reference.data["years"]["climate_full_years"]

worst = 0.0
for column, values in committed.items():
    if column not in computed:
        continue
    published = pd.Series(values, index=years)
    scale = max(published.abs().max(), 1e-12)
    worst = max(worst, float((computed.loc[years, column] - published).abs().max() / scale))

print(f"worst relative difference against committed s1.json: {worst:.2e}")
# Tolerance is 1e-3 rather than machine precision. The committed outputs were
# generated on Linux; on other platforms the aerosol forcing chain (soot_erf ->
# aerosol_erf -> the temperature integral) differs at the 1e-5 level, while ~77%
# of series still match bit-for-bit. A real model change moves whole families of
# outputs well past this bound, so the guard still does its job.
assert worst < 1e-3, "the central/T3/O3/F2 cell does not reproduce the published S1"

## Run the sweep

Results are persisted as one tidy frame of annual series, small enough to commit, so the
document can read them back without re-running anything.

In [ ]:
tidy = sweep.run_sweep()
path = sweep.write_results(tidy)
print(f"{len(tidy):,} rows written to {path.name}")

In [ ]:
summary = sweep.summarise(tidy, year=2050)
summary[
    [
        "traffic",
        "technology",
        "operations",
        "saf",
        "co2_emissions",
        "carbon_offset",
        "temperature_increase_from_aviation",
    ]
].sort_values("co2_emissions").head(12)

## Where the published scenarios sit in the grid

The spread over the whole grid is the quantity the report does not publish: how much of the
2050 outcome is settled by the lever combination rather than by the two illustrative choices.

In [ ]:
residual = summary.set_index(["traffic", "technology", "operations", "saf"])["co2_emissions"]

print("2050 residual CO2 across the grid [Mt]:")
print(f"  min    {residual.min():8.0f}   ({residual.idxmin()})")
print(f"  median {residual.median():8.0f}")
print(f"  max    {residual.max():8.0f}   ({residual.idxmax()})")
for name, cell in sweep.PUBLISHED_CELLS.items():
    print(f"  {name}     {residual.loc[cell]:8.0f}   ({cell})")

## The whole grid at once

Every one of the 108 combinations, drawn as a single translucent line, across four metrics: CO₂
emissions, final energy, energy intensity and carbon intensity. Individual runs rather than a
min/max envelope, because the grid is not one ordered family — it is four levers crossed, and the
density of the bundle is the point.

The published S1 and S2 are overlaid in black, so the two reported cases can be located inside the
spread they belong to.

The series start at the last observed year rather than 2019: the COVID collapse drives RPK down
without a matching drop in energy, so 2020 reads about 2.3 MJ/RPK against a 1.4 trend, and that
spike compresses both intensity panels into illegibility.

In [ ]:
figure, axes = sweep.plot_grid(color_by="saf")

Colouring by SAF separates the two carbon panels cleanly and does nothing at all to the two energy
panels — which is exactly right, and worth seeing: substituting the fuel changes what a joule
emits, not how many joules are burned.

Colouring the same figure by traffic shows the complement.

In [ ]:
figure, axes = sweep.plot_grid(color_by="traffic")

## Technology against operations, at central traffic

Grouped with the standard comparison plot: one envelope per technology variant, spanning the
three operations levels.

In [ ]:
processes = {}
groups = {}
for technology in sweep.TECHNOLOGY_LEVELS:
    groups[technology] = []
    for operations in sweep.OPERATIONS_LEVELS:
        name = f"{technology} - {operations}"
        process = sweep.build_process("central", technology, operations, "F2")
        process.compute()
        processes[name] = process
        groups[technology].append(name)

assembly = assemble_processes(processes)

In [ ]:
assembly.plot("co2_emissions_comparison", scenario_groups=groups, group_display="envelope")

In [ ]:
assembly.plot("temperature_increase_comparison", scenario_groups=groups, group_display="envelope")

## Per-mechanism climate decomposition

The published scenarios reach net zero CO2 while their warming contribution stays dominated by
non-CO2 terms, which is visible only once the mechanisms are separated.

In [ ]:
published = assemble_processes(
    {name: sweep.build_process(*cell) for name, cell in sweep.PUBLISHED_CELLS.items()}
)
published.compute_all()
published.plot("temperature_decomposition_comparison")